[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/evaluate_dataset_quality.ipynb)

# Evaluate Dataset Quality

Wait for evaluation to finish and read the before/after quality metrics for an adapted dataset — both the full result and the compact summary.

**Docs:** [Evaluate dataset quality](https://docs.adaptionlabs.ai/adaptive-data/evaluate-dataset-quality/)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

In [ ]:
%pip install -q adaption

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (🔑 in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

Point at a dataset you have already adapted by setting `DATASET_ID`. Leave it `None` to adapt a small sample dataset first, then evaluate that.

In [ ]:
DATASET_ID = None

## Step 1 — Make sure you have an adapted dataset

Evaluation compares the source and adapted rows, so it needs a completed adaptation run. When `DATASET_ID` is `None`, this builds a tiny one (`max_rows=100`).

In [ ]:
import time

if DATASET_ID is None:
    ds = client.datasets.create(
        source={
            "url": "https://huggingface.co/datasets/tatsu-lab/alpaca",
            "files": ["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"],
        },
    )
    DATASET_ID = ds.dataset_id

    while True:
        status = client.datasets.get_status(DATASET_ID)
        if status.status == "failed":
            raise RuntimeError(f"Ingestion failed: {status.error_data}")
        if status.row_count is not None:
            break
        time.sleep(5)

    client.datasets.adapt(
        DATASET_ID,
        column_mapping={
            "prompt": "instruction",
            "completion": "output",
            "context": ["input"],
        },
        job_specification={"max_rows": 100},
    )
    status = client.datasets.wait_for_completion(DATASET_ID, timeout=3600)
    if status.status == "failed":
        raise RuntimeError(f"Adaptation failed: {status.error_data}")

print("Evaluating dataset:", DATASET_ID)

## Step 2 — Wait for evaluation

Evaluation runs on its own schedule and can finish after adaptation. Poll `get_evaluation` with a deadline. Terminal states are `succeeded`, `failed`, and `skipped`.

In [ ]:
deadline = time.monotonic() + 600
while True:
    evaluation = client.datasets.get_evaluation(DATASET_ID)
    print("Evaluation status:", evaluation.status)
    if evaluation.status in ("succeeded", "failed", "skipped"):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Evaluation did not finish within 10 minutes")
    time.sleep(5)

## Step 3 — Read the quality result

On success, `quality` can include before/after scores on a 0–10 scale, letter grades, an improvement percentage, and an after-adaptation percentile.

In [ ]:
if evaluation.status == "failed":
    raise RuntimeError("Dataset evaluation failed")
elif evaluation.status == "skipped":
    print("Evaluation was skipped for this dataset")
elif evaluation.quality:
    q = evaluation.quality
    print(f"Score before: {q.score_before}")
    print(f"Score after:  {q.score_after}")
    print(f"Improvement:  {q.improvement_percent}%")
    print(q.model_dump(exclude_none=True))

## Step 4 — Or read the compact summary

If your app already fetches the dataset, use its `evaluation_summary` for headline metrics without a separate evaluation request.

In [ ]:
dataset = client.datasets.get(dataset_id=DATASET_ID)
if dataset.evaluation_summary:
    print(dataset.evaluation_summary.model_dump(exclude_none=True))
else:
    print("No evaluation summary yet.")

## Next steps

- To improve the score, [limit rows and estimate](https://docs.adaptionlabs.ai/adaptive-data/configure-adaptive-data/#limit-rows-and-estimate-a-run) while tuning recipes and brand controls.
- [Train a model with AutoScientist](https://docs.adaptionlabs.ai/autoscientist-quickstart/) once you are happy with quality.

## Clean up

In [ ]:
# client.datasets.delete(dataset_id=DATASET_ID)